# 01 - Coleta e padronização reprodutível dos fundos (v3)

Esta versão mantém `data/raw/` imutável e produz fundos processados com identificadores
estáveis derivados de SHA-256. A padronização usa recorte central quadrado, sem deformar
a razão de aspecto, e normalização robusta por percentis.

**Saídas:**
- `data/processed_v3/backgrounds_base/`
- `data/processed_v3/manifest_base_v3.csv`
- metadados de coleta em `data/raw/metadata_v3/`

A normalização é uma transformação computacional, não uma calibração fotométrica.


In [1]:
%pip install -q kaggle astroquery opencv-python-headless pillow tqdm pandas

from google.colab import drive, userdata
from pathlib import Path
from datetime import date, datetime, timezone, timedelta
import csv, gc, hashlib, json, math, os, shutil, time
import cv2
import numpy as np
import pandas as pd

# `force_remount` evita o estado FUSE quebrado do Colab
# (OSError 107: Transport endpoint is not connected).
drive.mount('/content/drive', force_remount=True)
BACKUP_DIR = Path('/content/drive/MyDrive/tcc-satellite-streaks')
RAW = Path('data/raw')
META = RAW / 'metadata_v3'
PROC = Path('data/processed_v3')
BASE = PROC / 'backgrounds_base'
for pasta in [RAW, META, PROC, BASE]:
    pasta.mkdir(parents=True, exist_ok=True)

DATA_EXECUCAO = datetime.now(timezone.utc).isoformat()
SEED_COLETA = 20260823
TAMANHO = 512

Mounted at /content/drive


## 1. Restaurar somente o dado bruto

O bruto é cumulativo e imutável; por isso a restauração pode ser incremental. Pastas
processadas não são restauradas aqui, evitando que artefatos antigos reapareçam.

In [2]:
# ASTA é teste externo intocado e NÃO é restaurado nesta etapa de fundos.
# A lista explícita também impede que outras pastas acidentais entrem no pipeline.
PASTAS_RAW_FUNDOS = ('hubble', 'jwst', 'sdss', 'apod', 'metadata_v3')

def copiar_raw_seletivo(origem_raiz, destino_raiz):
    origem_raiz, destino_raiz = Path(origem_raiz), Path(destino_raiz)
    copiadas = []
    for nome in PASTAS_RAW_FUNDOS:
        origem = origem_raiz / nome
        if not origem.is_dir():
            continue
        try:
            shutil.copytree(origem, destino_raiz / nome, dirs_exist_ok=True)
            copiadas.append(nome)
        except OSError as exc:
            raise RuntimeError(
                'O Google Drive desconectou durante a cópia. '
                'Reconecte o Drive, execute novamente a célula de configuração '
                'e depois repita esta célula; a cópia incremental pode ser retomada.'
            ) from exc
    return copiadas

origem_raw = BACKUP_DIR / 'data/raw'
if origem_raw.is_dir():
    copiadas = copiar_raw_seletivo(origem_raw, RAW)
    print('Pastas de fundos restauradas:', copiadas or 'nenhuma disponível')
    print('ASTA preservado no Drive e excluído desta etapa.')
else:
    print('Primeira execução: ainda não existe backup bruto.')

Pastas de fundos restauradas: ['hubble', 'jwst', 'sdss', 'apod']
ASTA preservado no Drive e excluído desta etapa.


## 2. Coleta opcional Kaggle

Estas coleções são preservadas como bruto, mas serão categoria B por padrão na auditoria,
pois contêm majoritariamente imagens icônicas processadas.

In [3]:
BAIXAR_KAGGLE = False  # altere para True somente se os arquivos ainda não estiverem no raw
if BAIXAR_KAGGLE:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    (RAW / 'hubble').mkdir(parents=True, exist_ok=True)
    (RAW / 'jwst').mkdir(parents=True, exist_ok=True)
    !kaggle datasets download -d redwankarimsony/top-100-hubble-telescope-images -p data/raw/hubble --unzip
    !kaggle datasets download -d goelyash/james-webb-telescope-images-original-size -p data/raw/jwst --unzip

## 3. Coleta SDSS determinística e com metadados

Os pontos candidatos são reproduzíveis. Cada FITS é identificado por coordenadas, e os
metadados preservam RA/Dec e a data da consulta.

In [4]:
from astropy import coordinates as coords
from astropy import units as u
from astroquery.sdss import SDSS

COLETAR_SDSS = False
META_SDSS = META / 'sdss_metadata_v3.csv'
N_CAMPOS_SDSS = 250

if COLETAR_SDSS:
    pasta = RAW / 'sdss'
    pasta.mkdir(parents=True, exist_ok=True)
    existentes = pd.read_csv(META_SDSS) if META_SDSS.exists() else pd.DataFrame()
    ids_existentes = set(existentes.get('id_consulta', pd.Series(dtype=str)).astype(str))
    linhas = existentes.to_dict('records') if len(existentes) else []
    rng = np.random.default_rng(SEED_COLETA)
    candidatos = [(rng.uniform(100, 260), rng.uniform(-5, 60)) for _ in range(N_CAMPOS_SDSS * 12)]

    for indice, (ra, dec) in enumerate(candidatos):
        if len(linhas) >= N_CAMPOS_SDSS:
            break
        id_consulta = f'{indice:05d}_{ra:.6f}_{dec:.6f}'
        if id_consulta in ids_existentes:
            continue
        pos = coords.SkyCoord(ra=ra*u.degree, dec=dec*u.degree, frame='icrs')
        try:
            encontrados = SDSS.query_region(pos, radius=2*u.arcmin, spectro=False)
            if encontrados is None or len(encontrados) == 0:
                continue
            imagens = SDSS.get_images(matches=encontrados[:1], band='r')
            if not imagens:
                continue
            nome = f'sdss_{id_consulta}.fits'
            imagens[0].writeto(pasta / nome, overwrite=False)
            linhas.append({
                'id_consulta': id_consulta, 'arquivo_original': nome,
                'ra_deg': ra, 'dec_deg': dec, 'banda': 'r',
                'url_origem': 'https://www.sdss.org/',
                'data_coleta_utc': DATA_EXECUCAO,
            })
            pd.DataFrame(linhas).to_csv(META_SDSS, index=False)
        except Exception as exc:
            print(f'Consulta {id_consulta} ignorada: {type(exc).__name__}')
    print(f'SDSS registrados: {len(linhas)}')

## 4. Coleta APOD com proveniência por imagem

In [5]:
import requests

COLETAR_APOD = False
META_APOD = META / 'apod_metadata_v3.csv'
N_DIAS_APOD = 100

if COLETAR_APOD:
    api_key = userdata.get('NASA_API_KEY')
    pasta = RAW / 'apod'
    pasta.mkdir(parents=True, exist_ok=True)
    existentes = pd.read_csv(META_APOD) if META_APOD.exists() else pd.DataFrame()
    datas_existentes = set(existentes.get('data_apod', pd.Series(dtype=str)).astype(str))
    linhas = existentes.to_dict('records') if len(existentes) else []
    hoje = date.today()
    for deslocamento in range(1, N_DIAS_APOD + 1):
        dia = (hoje - timedelta(days=deslocamento)).isoformat()
        if dia in datas_existentes:
            continue
        resposta = requests.get('https://api.nasa.gov/planetary/apod',
                                params={'api_key': api_key, 'date': dia}, timeout=30)
        resposta.raise_for_status()
        meta = resposta.json()
        if meta.get('media_type') != 'image':
            continue
        url = meta.get('hdurl') or meta.get('url')
        imagem = requests.get(url, timeout=60)
        imagem.raise_for_status()
        extensao = Path(url.split('?')[0]).suffix.lower()
        if extensao not in {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}:
            extensao = '.jpg'
        nome = f'apod_{dia}{extensao}'
        (pasta / nome).write_bytes(imagem.content)
        linhas.append({
            'data_apod': dia, 'arquivo_original': nome, 'titulo': meta.get('title'),
            'copyright': meta.get('copyright'), 'url_origem': url,
            'url_api': f'https://api.nasa.gov/planetary/apod?date={dia}',
            'data_coleta_utc': DATA_EXECUCAO,
        })
        pd.DataFrame(linhas).to_csv(META_APOD, index=False)
        time.sleep(0.5)
    print(f'APOD registrados: {len(linhas)}')

## 5. Padronização determinística e IDs estáveis

A classificação humana posterior será vinculada a `fonte + arquivo_original + sha256_raw`,
nunca a um contador sequencial.

In [6]:
from astropy.io import fits
from PIL import Image

EXT_IMAGEM = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}
FONTES = ['hubble', 'jwst', 'sdss', 'apod']
URL_FONTE = {
    'hubble': 'https://www.kaggle.com/datasets/redwankarimsony/top-100-hubble-telescope-images',
    'jwst': 'https://www.kaggle.com/datasets/goelyash/james-webb-telescope-images-original-size',
    'sdss': 'https://www.sdss.org/',
    'apod': 'https://apod.nasa.gov/apod/',
}
LICENCA_FONTE = {
    'hubble': 'Kaggle: copyright-authors; verificar crédito original NASA/ESA/Hubble',
    'jwst': 'Kaggle: CC0-1.0; preservar proveniência original',
    'sdss': 'SDSS Collaboration; uso público com atribuição',
    'apod': 'Crédito/licença por imagem; verificar autor individual',
}

def sha256_arquivo(caminho, bloco=1024*1024):
    h = hashlib.sha256()
    with open(caminho, 'rb') as arquivo:
        for parte in iter(lambda: arquivo.read(bloco), b''):
            h.update(parte)
    return h.hexdigest()

MAX_LADO_INTERMEDIARIO = 2048
MAX_PIXELS_RASTER_NAO_JPEG = 30_000_000

def ler_e_reduzir(caminho, fonte, tamanho=512):
    """Retorna uma imagem 2D pequena sem materializar o arquivo astronômico inteiro.

    FITS usa memmap, recorte central e subamostragem antes de virar float32. JPEG usa
    o decoder reduzido do Pillow. Raster enorme sem decoder reduzido é recusado de
    forma controlada, em vez de derrubar a sessão por falta de RAM.
    """
    if fonte == 'sdss' or caminho.suffix.lower() == '.fits':
        with fits.open(caminho, memmap=True, do_not_scale_image_data=True) as hdul:
            dados = hdul[0].data
            if dados is None:
                raise ValueError('FITS sem dados no HDU primário')
            while dados.ndim > 2:
                dados = dados[0]
            if dados.ndim != 2:
                raise ValueError(f'FITS com dimensionalidade não suportada: {dados.ndim}')
            altura, largura = dados.shape
            lado = min(altura, largura)
            y0, x0 = (altura-lado)//2, (largura-lado)//2
            passo = max(1, math.ceil(lado / MAX_LADO_INTERMEDIARIO))
            recorte = dados[y0:y0+lado:passo, x0:x0+lado:passo]
            reduzida = cv2.resize(
                np.asarray(recorte, dtype=np.float32),
                (tamanho, tamanho), interpolation=cv2.INTER_AREA
            )
        return reduzida

    with Image.open(caminho) as pil:
        formato = (pil.format or '').upper()
        if formato in {'JPEG', 'JPG', 'MPO'}:
            pil.draft('L', (MAX_LADO_INTERMEDIARIO, MAX_LADO_INTERMEDIARIO))
        largura, altura = pil.size
        if (largura * altura > MAX_PIXELS_RASTER_NAO_JPEG
                and formato not in {'JPEG', 'JPG', 'MPO'}):
            raise MemoryError(
                f'raster {largura}x{altura} ignorado para proteger a RAM; '
                'converter previamente para JPEG/PNG reduzido se for indispensável'
            )
        lado = min(largura, altura)
        x0, y0 = (largura-lado)//2, (altura-lado)//2
        recorte = pil.crop((x0, y0, x0+lado, y0+lado)).convert('L')
        reduzida = recorte.resize((tamanho, tamanho), Image.Resampling.LANCZOS)
        return np.asarray(reduzida, dtype=np.float32)

def normalizar_robusto(imagem, p_baixo=1.0, p_alto=99.8):
    finitos = imagem[np.isfinite(imagem)]
    if finitos.size == 0:
        raise ValueError('imagem sem pixels finitos')
    baixo, alto = np.percentile(finitos, [p_baixo, p_alto])
    if alto <= baixo:
        raise ValueError('imagem sem faixa dinâmica utilizável')
    return np.clip((imagem - baixo) / (alto - baixo), 0, 1) * 255

tarefas = []
for fonte in FONTES:
    pasta = RAW / fonte
    if not pasta.is_dir():
        continue
    for caminho in sorted(p for p in pasta.rglob('*') if p.is_file()):
        if fonte == 'sdss' and caminho.suffix.lower() != '.fits':
            continue
        if fonte != 'sdss' and caminho.suffix.lower() not in EXT_IMAGEM:
            continue
        tarefas.append((fonte, caminho))

linhas = []
falhas = []
hashes_vistos = {}
nomes_esperados = set()
# Cache versionado no Drive: uma queda posterior não obriga a reprocessar arquivos já concluídos.
CHECKPOINT_BASE = (BACKUP_DIR / 'data/processed_v3/checkpoints'
                   / 'preprocess_safe_v3_1' / 'backgrounds_base')
CHECKPOINT_BASE.mkdir(parents=True, exist_ok=True)
for indice, (fonte, caminho) in enumerate(tarefas, start=1):
    pequena = None
    processada = None
    try:
        hash_raw = sha256_arquivo(caminho)
        if hash_raw in hashes_vistos:
            raise ValueError(
                f'conteúdo duplicado de {hashes_vistos[hash_raw]}; '
                'segunda cópia excluída do manifest'
            )
        id_fundo = f'{fonte}_{hash_raw[:16]}'
        nome_final = f'{id_fundo}.png'
        destino = BASE / nome_final
        checkpoint = CHECKPOINT_BASE / nome_final
        recuperado = False
        if checkpoint.is_file():
            try:
                shutil.copy2(checkpoint, destino)
                teste_cache = cv2.imread(str(destino), cv2.IMREAD_GRAYSCALE)
                recuperado = teste_cache is not None and teste_cache.shape == (TAMANHO, TAMANHO)
                teste_cache = None
            except OSError as exc:
                print(f'Aviso: checkpoint indisponível para {nome_final}: {exc}')
                recuperado = False

        if not recuperado:
            pequena = ler_e_reduzir(caminho, fonte, TAMANHO)
            processada = normalizar_robusto(pequena).astype(np.uint8)
            if not cv2.imwrite(str(destino), processada):
                raise IOError('cv2.imwrite retornou False')
            try:
                shutil.copy2(destino, checkpoint)
            except OSError as exc:
                # O resultado local continua válido; apenas a retomada fica indisponível.
                print(f'Aviso: não foi possível salvar checkpoint de {nome_final}: {exc}')
        hash_processado = sha256_arquivo(destino)
        nomes_esperados.add(nome_final)
        linhas.append({
            'id_fundo': id_fundo, 'fonte': fonte,
            'arquivo_original': caminho.name,
            'caminho_raw_relativo': caminho.relative_to(RAW).as_posix(),
            'arquivo_final': nome_final, 'sha256_raw': hash_raw,
            'sha256_processado': hash_processado,
            'url_fonte': URL_FONTE[fonte], 'licenca_fonte': LICENCA_FONTE[fonte],
            'normalizacao': 'percentis_1_99.8_apos_reducao_segura',
            'transformacao_geometrica': 'center_crop_square_resize_512',
        })
        hashes_vistos[hash_raw] = caminho.relative_to(RAW).as_posix()
    except Exception as exc:
        falhas.append({
            'fonte': fonte,
            'arquivo_original': caminho.name,
            'caminho_raw_relativo': caminho.relative_to(RAW).as_posix(),
            'tipo_erro': type(exc).__name__,
            'motivo': str(exc)[:500],
        })
        print(f'IGNORADO {fonte}/{caminho.name}: {type(exc).__name__}: {exc}')
    finally:
        # Impede acúmulo de buffers de decodificação entre arquivos grandes.
        pequena = None
        processada = None
        if indice % 10 == 0:
            gc.collect()
            print(f'Processados {indice}/{len(tarefas)}; válidos={len(linhas)}; ignorados={len(falhas)}')

# Remove somente sobras da pasta processada v3; o bruto nunca é tocado.
for caminho in BASE.glob('*'):
    if caminho.is_file() and caminho.name not in nomes_esperados:
        caminho.unlink()

if not linhas:
    raise RuntimeError('Nenhum fundo pôde ser processado; consulte manifest_falhas_v3.csv.')
df_base = pd.DataFrame(linhas).sort_values(['fonte', 'id_fundo']).reset_index(drop=True)
assert df_base['id_fundo'].is_unique
assert df_base['sha256_raw'].is_unique, 'Duplicata de conteúdo bruto encontrada; revisar antes de seguir.'
assert set(p.name for p in BASE.glob('*.png')) == set(df_base['arquivo_final'])
df_base.to_csv(PROC / 'manifest_base_v3.csv', index=False)
pd.DataFrame(falhas, columns=[
    'fonte', 'arquivo_original', 'caminho_raw_relativo', 'tipo_erro', 'motivo'
]).to_csv(PROC / 'manifest_falhas_v3.csv', index=False)
print(pd.crosstab(index=df_base['fonte'], columns='quantidade'))
print(f'Total processado: {len(df_base)}')
print(f'Total ignorado com justificativa: {len(falhas)}')

IGNORADO hubble/heic0406a.tif: MemoryError: raster 6200x6200 ignorado para proteger a RAM; converter previamente para JPEG/PNG reduzido se for indispensável


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (91414305 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


IGNORADO hubble/heic0506a.tif: MemoryError: raster 11477x7965 ignorado para proteger a RAM; converter previamente para JPEG/PNG reduzido se for indispensável
IGNORADO hubble/heic0506b.tif: MemoryError: raster 3857x7804 ignorado para proteger a RAM; converter previamente para JPEG/PNG reduzido se for indispensável
Processados 10/451; válidos=7; ignorados=3
IGNORADO hubble/heic0601a.tif: DecompressionBombError: Image size (324000000 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack.
IGNORADO hubble/heic0602a.tif: DecompressionBombError: Image size (196437984 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack.
IGNORADO hubble/heic0604a.tif: MemoryError: raster 9500x7400 ignorado para proteger a RAM; converter previamente para JPEG/PNG reduzido se for indispensável
IGNORADO hubble/heic0707a.tif: DecompressionBombError: Image size (423414686 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack.
Proce

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (123027575 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


IGNORADO jwst/Cosmic Cliffs in the Carina Nebula.tif: MemoryError: raster 14575x8441 ignorado para proteger a RAM; converter previamente para JPEG/PNG reduzido se for indispensável


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (153518328 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


IGNORADO jwst/Stephan Quintet.tif: MemoryError: raster 12654x12132 ignorado para proteger a RAM; converter previamente para JPEG/PNG reduzido se for indispensável


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (122540826 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


IGNORADO jwst/Tarantula Nebula.tif: MemoryError: raster 14557x8418 ignorado para proteger a RAM; converter previamente para JPEG/PNG reduzido se for indispensável
Processados 110/451; válidos=88; ignorados=22
IGNORADO jwst/Cosmic Cliffs in the Carina Nebula.tif: MemoryError: raster 11264x3904 ignorado para proteger a RAM; converter previamente para JPEG/PNG reduzido se for indispensável
IGNORADO jwst/Stephan Quintet.tif: MemoryError: raster 12654x12132 ignorado para proteger a RAM; converter previamente para JPEG/PNG reduzido se for indispensável
IGNORADO jwst/MIRI.png: MemoryError: raster 4988x16954 ignorado para proteger a RAM; converter previamente para JPEG/PNG reduzido se for indispensável
IGNORADO jwst/NIRCam.png: MemoryError: raster 4989x14594 ignorado para proteger a RAM; converter previamente para JPEG/PNG reduzido se for indispensável
Processados 120/451; válidos=94; ignorados=26
Processados 130/451; válidos=104; ignorados=26
Processados 140/451; válidos=114; ignorados=26
Pro

## Validação

In [7]:
# ============================================================
# GATE LOCAL — EXECUTAR ENTRE AS SEÇÕES 5 E 6
# Somente leitura. Não altera arquivos.
# ============================================================

from pathlib import Path
import hashlib
import json
import re

import cv2
import numpy as np
import pandas as pd


RAW = Path("data/raw")
PROC = Path("data/processed_v3")
BASE = PROC / "backgrounds_base"
MANIFESTO = PROC / "manifest_base_v3.csv"
MANIFESTO_FALHAS = PROC / "manifest_falhas_v3.csv"

FONTES_PERMITIDAS = {"hubble", "jwst", "sdss", "apod"}

COLUNAS = {
    "id_fundo",
    "fonte",
    "arquivo_original",
    "caminho_raw_relativo",
    "arquivo_final",
    "sha256_raw",
    "sha256_processado",
    "url_fonte",
    "licenca_fonte",
    "normalizacao",
    "transformacao_geometrica",
}

HASH_RE = re.compile(r"^[0-9a-f]{64}$")

erros = []
avisos = []


def erro(texto):
    erros.append(str(texto))


def aviso(texto):
    avisos.append(str(texto))


def sha256(caminho, bloco=1024 * 1024):
    h = hashlib.sha256()
    with open(caminho, "rb") as arquivo:
        for parte in iter(lambda: arquivo.read(bloco), b""):
            h.update(parte)
    return h.hexdigest()


# ------------------------------------------------------------
# 1. Artefatos obrigatórios
# ------------------------------------------------------------

for nome, caminho, tipo in [
    ("raw local", RAW, "pasta"),
    ("backgrounds_base", BASE, "pasta"),
    ("manifest_base_v3.csv", MANIFESTO, "arquivo"),
    ("manifest_falhas_v3.csv", MANIFESTO_FALHAS, "arquivo"),
]:
    existe = caminho.is_dir() if tipo == "pasta" else caminho.is_file()
    if not existe:
        erro(f"Ausente: {nome} — {caminho}")

if erros:
    print(json.dumps({
        "status": "REPROVADO",
        "erros": erros,
        "avisos": avisos,
    }, ensure_ascii=False, indent=2))
    raise RuntimeError("Artefatos obrigatórios ausentes.")


# ------------------------------------------------------------
# 2. Manifest principal
# ------------------------------------------------------------

df = pd.read_csv(MANIFESTO).fillna("")

if df.empty:
    erro("O manifest principal está vazio.")

faltantes = sorted(COLUNAS - set(df.columns))
if faltantes:
    erro(f"Colunas ausentes no manifest: {faltantes}")

if faltantes:
    print(json.dumps({
        "status": "REPROVADO",
        "erros": erros,
        "avisos": avisos,
    }, ensure_ascii=False, indent=2))
    raise RuntimeError("Estrutura do manifest inválida.")

for coluna in COLUNAS:
    df[coluna] = df[coluna].astype(str).str.strip()

campos_vazios = {
    coluna: int((df[coluna] == "").sum())
    for coluna in COLUNAS
    if (df[coluna] == "").any()
}

if campos_vazios:
    erro(f"Campos obrigatórios vazios: {campos_vazios}")

for coluna in [
    "id_fundo",
    "arquivo_final",
    "sha256_raw",
    "sha256_processado",
]:
    quantidade = int(df[coluna].duplicated(keep=False).sum())
    if quantidade:
        erro(f"{quantidade} ocorrências duplicadas em {coluna}.")

fontes_invalidas = sorted(set(df["fonte"]) - FONTES_PERMITIDAS)
if fontes_invalidas:
    erro(f"Fontes não permitidas: {fontes_invalidas}")

contagem_fontes = {
    str(fonte): int(quantidade)
    for fonte, quantidade in df["fonte"].value_counts().sort_index().items()
}

if contagem_fontes.get("sdss", 0) == 0:
    erro("Nenhum fundo SDSS foi processado.")

if len(df) < 100:
    aviso(
        f"Somente {len(df)} fundos foram processados. "
        "A suficiência será confirmada após a auditoria de domínio."
    )

if not df["sha256_raw"].map(lambda x: bool(HASH_RE.fullmatch(x))).all():
    erro("Existem hashes raw inválidos.")

if not df["sha256_processado"].map(
    lambda x: bool(HASH_RE.fullmatch(x))
).all():
    erro("Existem hashes processados inválidos.")

if set(df["normalizacao"]) != {
    "percentis_1_99.8_apos_reducao_segura"
}:
    erro(
        "O manifest contém normalização diferente da versão segura esperada: "
        f"{sorted(set(df['normalizacao']))}"
    )

if set(df["transformacao_geometrica"]) != {
    "center_crop_square_resize_512"
}:
    erro(
        "Transformação geométrica inesperada: "
        f"{sorted(set(df['transformacao_geometrica']))}"
    )


# ------------------------------------------------------------
# 3. Correspondência manifest ↔ pasta
# ------------------------------------------------------------

esperados = set(df["arquivo_final"])

encontrados = {
    p.name
    for p in BASE.iterdir()
    if p.is_file() and p.suffix.lower() == ".png"
}

ausentes = sorted(esperados - encontrados)
orfaos = sorted(encontrados - esperados)

if ausentes:
    erro(f"{len(ausentes)} imagens processadas estão ausentes: {ausentes[:10]}")

if orfaos:
    erro(f"{len(orfaos)} imagens órfãs foram encontradas: {orfaos[:10]}")


# ------------------------------------------------------------
# 4. Verificação individual e hashes
# ------------------------------------------------------------

raw_ausentes = []
hashes_raw_errados = []
hashes_processados_errados = []
imagens_invalidas = []
baixo_contraste = []
muito_saturadas = []

desvios = []
percentuais_zero = []
percentuais_255 = []

print(f"Verificando {len(df)} registros...")

for numero, linha in df.iterrows():
    fonte = linha["fonte"]
    hash_raw_manifest = linha["sha256_raw"]

    id_esperado = f"{fonte}_{hash_raw_manifest[:16]}"
    arquivo_esperado = f"{id_esperado}.png"

    if linha["id_fundo"] != id_esperado:
        erro(
            f"ID inconsistente: {linha['id_fundo']} deveria ser {id_esperado}"
        )

    if linha["arquivo_final"] != arquivo_esperado:
        erro(
            f"Nome inconsistente: {linha['arquivo_final']} "
            f"deveria ser {arquivo_esperado}"
        )

    caminho_relativo = Path(linha["caminho_raw_relativo"])

    if (
        caminho_relativo.is_absolute()
        or ".." in caminho_relativo.parts
        or len(caminho_relativo.parts) < 2
    ):
        erro(f"Caminho raw inválido: {caminho_relativo}")
        continue

    if caminho_relativo.parts[0] != fonte:
        erro(
            f"Fonte {fonte} não corresponde ao caminho {caminho_relativo}"
        )

    if "asta" in str(caminho_relativo).lower():
        erro(f"ASTA apareceu no dataset de fundos: {caminho_relativo}")

    raw = RAW / caminho_relativo
    processada = BASE / linha["arquivo_final"]

    if not raw.is_file():
        raw_ausentes.append(str(caminho_relativo))
    else:
        try:
            if sha256(raw) != hash_raw_manifest:
                hashes_raw_errados.append(linha["id_fundo"])
        except Exception as exc:
            hashes_raw_errados.append(
                f"{linha['id_fundo']}: {type(exc).__name__}: {exc}"
            )

    if processada.is_file():
        try:
            if sha256(processada) != linha["sha256_processado"]:
                hashes_processados_errados.append(linha["id_fundo"])

            imagem = cv2.imread(
                str(processada),
                cv2.IMREAD_UNCHANGED,
            )

            if imagem is None:
                imagens_invalidas.append(
                    f"{linha['id_fundo']}: arquivo ilegível"
                )
            elif imagem.ndim != 2:
                imagens_invalidas.append(
                    f"{linha['id_fundo']}: ndim={imagem.ndim}"
                )
            elif imagem.shape != (512, 512):
                imagens_invalidas.append(
                    f"{linha['id_fundo']}: shape={imagem.shape}"
                )
            elif imagem.dtype != np.uint8:
                imagens_invalidas.append(
                    f"{linha['id_fundo']}: dtype={imagem.dtype}"
                )
            else:
                desvio = float(np.std(imagem))
                pct_zero = float(np.mean(imagem == 0))
                pct_255 = float(np.mean(imagem == 255))

                desvios.append(desvio)
                percentuais_zero.append(pct_zero)
                percentuais_255.append(pct_255)

                if desvio < 2:
                    baixo_contraste.append(linha["id_fundo"])

                if pct_zero > 0.95 or pct_255 > 0.95:
                    muito_saturadas.append(linha["id_fundo"])

            imagem = None

        except Exception as exc:
            imagens_invalidas.append(
                f"{linha['id_fundo']}: {type(exc).__name__}: {exc}"
            )

    if (numero + 1) % 50 == 0 or numero + 1 == len(df):
        print(f"  {numero + 1}/{len(df)} verificados")

if raw_ausentes:
    erro(f"{len(raw_ausentes)} arquivos raw estão ausentes: {raw_ausentes[:10]}")

if hashes_raw_errados:
    erro(
        f"{len(hashes_raw_errados)} hashes raw divergentes: "
        f"{hashes_raw_errados[:10]}"
    )

if hashes_processados_errados:
    erro(
        f"{len(hashes_processados_errados)} hashes processados divergentes: "
        f"{hashes_processados_errados[:10]}"
    )

if imagens_invalidas:
    erro(
        f"{len(imagens_invalidas)} imagens processadas inválidas: "
        f"{imagens_invalidas[:10]}"
    )

if baixo_contraste:
    aviso(
        f"{len(baixo_contraste)} imagens têm contraste muito baixo: "
        f"{baixo_contraste[:15]}"
    )

if muito_saturadas:
    aviso(
        f"{len(muito_saturadas)} imagens estão excessivamente saturadas: "
        f"{muito_saturadas[:15]}"
    )


# ------------------------------------------------------------
# 5. Manifest de falhas controladas
# ------------------------------------------------------------

df_falhas = pd.read_csv(MANIFESTO_FALHAS).fillna("")

colunas_falhas = {
    "fonte",
    "arquivo_original",
    "caminho_raw_relativo",
    "tipo_erro",
    "motivo",
}

faltantes_falhas = sorted(colunas_falhas - set(df_falhas.columns))

if faltantes_falhas:
    erro(
        f"Colunas ausentes no manifest de falhas: {faltantes_falhas}"
    )

resumo_falhas = {}

if not faltantes_falhas:
    resumo_falhas = {
        str(tipo): int(quantidade)
        for tipo, quantidade in
        df_falhas["tipo_erro"].value_counts().sort_index().items()
    }

    sobreposicao = (
        set(df["caminho_raw_relativo"])
        & set(df_falhas["caminho_raw_relativo"].astype(str))
    )

    if sobreposicao:
        erro(
            "Arquivos aparecem como aprovados e falhos simultaneamente: "
            f"{sorted(sobreposicao)[:10]}"
        )

    if len(df_falhas):
        aviso(
            f"{len(df_falhas)} arquivos foram recusados de forma controlada: "
            f"{resumo_falhas}"
        )


# ------------------------------------------------------------
# 6. Resultado
# ------------------------------------------------------------

estatisticas = {}

if desvios:
    estatisticas = {
        "desvio_min": round(float(np.min(desvios)), 4),
        "desvio_mediano": round(float(np.median(desvios)), 4),
        "desvio_max": round(float(np.max(desvios)), 4),
        "percentual_zero_max": round(
            float(np.max(percentuais_zero)) * 100, 4
        ),
        "percentual_255_max": round(
            float(np.max(percentuais_255)) * 100, 4
        ),
    }

NOTEBOOK_01_APROVADO = len(erros) == 0

relatorio = {
    "status": (
        "APROVADO PARA EXECUTAR A SEÇÃO 6"
        if NOTEBOOK_01_APROVADO
        else "REPROVADO — NÃO EXECUTAR A SEÇÃO 6"
    ),
    "total_processado": int(len(df)),
    "contagem_por_fonte": contagem_fontes,
    "total_falhas_controladas": int(len(df_falhas)),
    "falhas_por_tipo": resumo_falhas,
    "imagens_baixo_contraste": len(baixo_contraste),
    "imagens_muito_saturadas": len(muito_saturadas),
    "estatisticas": estatisticas,
    "quantidade_erros": len(erros),
    "quantidade_avisos": len(avisos),
    "erros": erros,
    "avisos": avisos,
}

print("\n" + "=" * 72)
print("RELATÓRIO PRÉ-BACKUP PARA COPIAR")
print("=" * 72)
print(json.dumps(relatorio, ensure_ascii=False, indent=2))
print("=" * 72)

if NOTEBOOK_01_APROVADO:
    print("\n✅ Pode executar a seção 6.")
else:
    print("\n❌ Não execute a seção 6. Envie este relatório para correção.")

Verificando 424 registros...
  50/424 verificados
  100/424 verificados
  150/424 verificados
  200/424 verificados
  250/424 verificados
  300/424 verificados
  350/424 verificados
  400/424 verificados
  424/424 verificados

RELATÓRIO PRÉ-BACKUP PARA COPIAR
{
  "status": "APROVADO PARA EXECUTAR A SEÇÃO 6",
  "total_processado": 424,
  "contagem_por_fonte": {
    "apod": 86,
    "hubble": 80,
    "jwst": 8,
    "sdss": 250
  },
  "total_falhas_controladas": 27,
  "falhas_por_tipo": {
    "DecompressionBombError": 3,
    "MemoryError": 24
  },
  "imagens_baixo_contraste": 0,
  "imagens_muito_saturadas": 0,
  "estatisticas": {
    "desvio_min": 13.6051,
    "desvio_mediano": 16.2822,
    "desvio_max": 99.8968,
    "percentual_zero_max": 74.0726,
    "percentual_255_max": 2.9922
  },
  "quantidade_erros": 0,
  "quantidade_avisos": 1,
  "erros": [],
  "avisos": [
    "27 arquivos foram recusados de forma controlada: {'DecompressionBombError': 3, 'MemoryError': 24}"
  ]
}

✅ Pode executar 

## 6. Backup

`backgrounds_base` é reconstruível. O destino é espelhado por nome para não preservar
arquivos órfãos. O raw continua incremental e imutável.

In [8]:
def espelhar_pasta_plana(origem, destino):
    origem, destino = Path(origem), Path(destino)
    destino.mkdir(parents=True, exist_ok=True)
    locais = {p.name: p for p in origem.iterdir() if p.is_file()}
    remotos = {p.name: p for p in destino.iterdir() if p.is_file()}
    if not locais:
        raise RuntimeError('Espelhamento bloqueado: origem vazia.')
    for nome, caminho in locais.items():
        if nome not in remotos or sha256_arquivo(caminho) != sha256_arquivo(remotos[nome]):
            shutil.copy2(caminho, destino / nome)
    for nome, caminho in remotos.items():
        if nome not in locais:
            caminho.unlink()
    assert set(locais) == {p.name for p in destino.iterdir() if p.is_file()}

# Backup incremental somente das fontes de fundos. Nunca toca em data/raw/asta_real.
copiadas = copiar_raw_seletivo(RAW, BACKUP_DIR / 'data/raw')
print('Pastas brutas atualizadas no Drive:', copiadas)
destino_proc = BACKUP_DIR / 'data/processed_v3'
espelhar_pasta_plana(BASE, destino_proc / 'backgrounds_base')
destino_proc.mkdir(parents=True, exist_ok=True)
shutil.copy2(PROC / 'manifest_base_v3.csv', destino_proc / 'manifest_base_v3.csv')
shutil.copy2(PROC / 'manifest_falhas_v3.csv', destino_proc / 'manifest_falhas_v3.csv')
print('Backup v3 concluído e verificado.')

Pastas brutas atualizadas no Drive: ['hubble', 'jwst', 'sdss', 'apod', 'metadata_v3']
Backup v3 concluído e verificado.


## 7. Validação Backup

In [9]:
# ============================================================
# VERIFICAÇÃO DO BACKUP — EXECUTAR DEPOIS DA SEÇÃO 6
# Somente leitura.
# ============================================================

from pathlib import Path
import hashlib
import json
import pandas as pd


PROC = Path("data/processed_v3")
BASE = PROC / "backgrounds_base"
MANIFESTO = PROC / "manifest_base_v3.csv"
MANIFESTO_FALHAS = PROC / "manifest_falhas_v3.csv"

DRIVE_PROC = Path(
    "/content/drive/MyDrive/tcc-satellite-streaks/data/processed_v3"
)
DRIVE_BASE = DRIVE_PROC / "backgrounds_base"
DRIVE_MANIFESTO = DRIVE_PROC / "manifest_base_v3.csv"
DRIVE_FALHAS = DRIVE_PROC / "manifest_falhas_v3.csv"

erros = []


def sha256(caminho, bloco=1024 * 1024):
    h = hashlib.sha256()
    with open(caminho, "rb") as arquivo:
        for parte in iter(lambda: arquivo.read(bloco), b""):
            h.update(parte)
    return h.hexdigest()


df = pd.read_csv(MANIFESTO).fillna("")
esperados = set(df["arquivo_final"])

if not DRIVE_BASE.is_dir():
    erros.append(f"Pasta ausente no Drive: {DRIVE_BASE}")
    encontrados = set()
else:
    encontrados = {
        p.name
        for p in DRIVE_BASE.iterdir()
        if p.is_file() and p.suffix.lower() == ".png"
    }

faltando = sorted(esperados - encontrados)
orfaos = sorted(encontrados - esperados)

if faltando:
    erros.append(
        f"{len(faltando)} imagens ausentes no Drive: {faltando[:10]}"
    )

if orfaos:
    erros.append(
        f"{len(orfaos)} imagens órfãs no Drive: {orfaos[:10]}"
    )

divergentes = []

if not faltando:
    for numero, linha in df.iterrows():
        remoto = DRIVE_BASE / linha["arquivo_final"]

        try:
            if sha256(remoto) != linha["sha256_processado"]:
                divergentes.append(linha["id_fundo"])
        except Exception as exc:
            erros.append(
                f"Falha ao ler {remoto.name}: "
                f"{type(exc).__name__}: {exc}"
            )
            break

        if (numero + 1) % 50 == 0 or numero + 1 == len(df):
            print(f"{numero + 1}/{len(df)} hashes do Drive verificados")

if divergentes:
    erros.append(
        f"{len(divergentes)} hashes divergentes no Drive: "
        f"{divergentes[:10]}"
    )

for local, remoto, nome in [
    (MANIFESTO, DRIVE_MANIFESTO, "manifest_base_v3.csv"),
    (MANIFESTO_FALHAS, DRIVE_FALHAS, "manifest_falhas_v3.csv"),
]:
    if not remoto.is_file():
        erros.append(f"{nome} ausente no Drive.")
    elif sha256(local) != sha256(remoto):
        erros.append(f"{nome} local e remoto são diferentes.")

relatorio = {
    "status": (
        "BACKUP APROVADO"
        if not erros
        else "BACKUP REPROVADO"
    ),
    "arquivos_esperados": len(esperados),
    "arquivos_encontrados_no_drive": len(encontrados),
    "arquivos_ausentes": len(faltando),
    "arquivos_orfaos": len(orfaos),
    "hashes_divergentes": len(divergentes),
    "erros": erros,
}

print("\n" + "=" * 72)
print("RELATÓRIO PÓS-BACKUP PARA COPIAR")
print("=" * 72)
print(json.dumps(relatorio, ensure_ascii=False, indent=2))
print("=" * 72)

if erros:
    print("\n❌ Não avance para o Notebook 02.")
else:
    print("\n✅ Notebook 01 e backup aprovados.")

50/424 hashes do Drive verificados
100/424 hashes do Drive verificados
150/424 hashes do Drive verificados
200/424 hashes do Drive verificados
250/424 hashes do Drive verificados
300/424 hashes do Drive verificados
350/424 hashes do Drive verificados
400/424 hashes do Drive verificados
424/424 hashes do Drive verificados

RELATÓRIO PÓS-BACKUP PARA COPIAR
{
  "status": "BACKUP APROVADO",
  "arquivos_esperados": 424,
  "arquivos_encontrados_no_drive": 424,
  "arquivos_ausentes": 0,
  "arquivos_orfaos": 0,
  "hashes_divergentes": 0,
  "erros": []
}

✅ Notebook 01 e backup aprovados.
